In [ ]:
from pathlib import Path
import numpy as np
from lisaorbits import OEMOrbits
# ============================================================
# Explicit settings
# ============================================================

BASE_DIR = Path.cwd()
INPUT_DIR = BASE_DIR / "inputs"
INPUT_DIR.mkdir(parents=True, exist_ok=True)

WAVEFORM_FILE = INPUT_DIR / "waveform_aligned.npz"
ORBITS_FILE = INPUT_DIR / "orbits.h5"

# Choose one:
#   "esa-trailing" = constellation trails Earth by 20 degrees
#   "esa-leading"  = constellation leads Earth by 20 degrees
ORBIT_NAME = "esa-trailing"

# Orbit-file sampling.
# This does NOT need to match the waveform sampling dt.
# It is the interpolation grid stored in the orbit HDF5 file.
DT_ORBIT = 600.0
ORBIT_DURATION_FACTOR = 5.0


def read_waveform_duration(waveform_file):
    data = np.load(waveform_file)
    t = np.asarray(data["t"], dtype=float)

    if len(t) < 2:
        raise ValueError("Waveform time array must have at least two samples.")

    dt = float(np.median(np.diff(t)))
    duration = dt * (len(t) - 1)
    return duration, dt, len(t)


def main():
    print(f"Creating realistic LISA orbit file using {ORBIT_NAME}")

    # This uses ESA-provided OEM orbits included/downloaded by lisaorbits.
    # The underlying spacecraft positions are interpolated from OEM files.
    orbits = OEMOrbits.from_included(
        ORBIT_NAME,
        version="latest",
    )

    print(f"ESA orbit starts at t_start = {orbits.t_start:.3f} s")
    print(f"ESA orbit ends   at t_end   = {orbits.t_end:.3f} s")

    waveform_duration, waveform_dt, waveform_size = read_waveform_duration(WAVEFORM_FILE)
    requested_duration = ORBIT_DURATION_FACTOR * waveform_duration
    size = int(np.ceil(requested_duration / DT_ORBIT)) + 1
    actual_duration = DT_ORBIT * (size - 1)
    pre_post_margin = 0.5 * (actual_duration - waveform_duration)

    print("\nWaveform timing:")
    print(f"  size     = {waveform_size}")
    print(f"  dt       = {waveform_dt} s")
    print(f"  duration = {waveform_duration} s")

    print("\nOrbit-file timing:")
    print(f"  dt       = {DT_ORBIT} s")
    print(f"  duration factor = {ORBIT_DURATION_FACTOR}x science segment")
    print(f"  pre/post margin = {pre_post_margin} s")
    print(f"  requested duration = {requested_duration} s")
    print(f"  actual duration    = {actual_duration} s")
    print(f"  size     = {size}")

    # Use the default initial time of the OEM orbit. The response notebook
    # centers the burn-in plus science segment inside this generated span.
    t0 = "init"

    if ORBITS_FILE.exists():
        ORBITS_FILE.unlink()

    orbits.write(
        str(ORBITS_FILE),
        t0=t0,
        dt=DT_ORBIT,
        size=size,
        mode="x",
    )

    print(f"Wrote orbit file:")
    print(f"  {ORBITS_FILE}")

    # Quick sanity check on light travel times
    t = np.linspace(orbits.t_init, orbits.t_init + 5 * DT_ORBIT, 6)
    links = [12, 23, 31, 13, 32, 21]

    ltt = orbits.compute_ltt(t, links)
    ltt_dot = orbits.compute_ltt_derivative(t, links)

    print("\nExample light-travel times [s]:")
    print(ltt)

    print("\nExample light-travel-time derivatives [s/s]:")
    print(ltt_dot)


if __name__ == "__main__":
    main()
